In [ ]:
!pip install sentence-transformers faiss-cpu pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
df = pd.read_csv(
    "MASTER_FINANCIAL_CORPUS_V1.csv"
)
print(df.shape)
df.head()

(16893, 4)


,Doc_ID,Title,FinBERT_Label,Source
0,1,spicejet to issue 6.4 crore warrants to promoters,neutral,SEntFiN
1,2,mmtc q2 net loss at rs 10.4 crore,neutral,SEntFiN
2,3,"mid-cap funds can deliver more, stay put: experts",positive,SEntFiN
3,4,mid caps now turn into market darlings,positive,SEntFiN
4,5,"market seeing patience, if not conviction: pra...",neutral,SEntFiN


In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
embeddings = model.encode(
    df["Title"].tolist(),
    batch_size=64,
    show_progress_bar=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/264 [00:00<?, ?it/s]

In [ ]:
print(embeddings.shape)

(16893, 384)


In [ ]:
import faiss
import numpy as np
embeddings = np.array(
    embeddings,
    dtype=np.float32
)
faiss.normalize_L2(
    embeddings
)
index = faiss.IndexFlatIP(
    embeddings.shape[1]
)
index.add(
    embeddings
)
print(
    index.ntotal
)

16893


In [ ]:
query = "Infosys divident"

query_embedding = model.encode(
    [query]
)

query_embedding = np.array(
    query_embedding,
    dtype=np.float32
)

faiss.normalize_L2(
    query_embedding
)

D, I = index.search(
    query_embedding,
    5
)

In [ ]:
for idx in I[0]:
    print(
        df.iloc[idx]["Title"]
    )
    print(
        df.iloc[idx]["FinBERT_Label"]
    )
    print()

dividend - infosys limited has informed the exchange that board of directors at its meeting held on april 18, 2024, recommended final dividend of 28 per equity share.
positive

dividend updates - infosys limited has informed the exchange about dividend updates
positive

stay away from infosys at current levels: pramod gubbi
negative

do not see too much downside for infosys: deven choksey
neutral

lic trims stake in infosys to 3.25%; garners rs 850 crore
negative



In [ ]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
df = pd.read_csv("/content/MASTER_FINANCIAL_CORPUS_V1.csv")
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["FinBERT_Label"],
    random_state=42
)

print("Train:", train_df.shape)
print("Test :", test_df.shape)
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
train_embeddings = model.encode(
    train_df["Title"].tolist(),
    batch_size=64,
    show_progress_bar=True
)

train_embeddings = np.array(
    train_embeddings,
    dtype=np.float32
)

faiss.normalize_L2(train_embeddings)
index = faiss.IndexFlatIP(
    train_embeddings.shape[1]
)

index.add(train_embeddings)

print("Vectors Stored:", index.ntotal)
test_embeddings = model.encode(
    test_df["Title"].tolist(),
    batch_size=64,
    show_progress_bar=True
)

test_embeddings = np.array(
    test_embeddings,
    dtype=np.float32
)

faiss.normalize_L2(test_embeddings)
K = 5

D, I = index.search(
    test_embeddings,
    K
)
train_labels = train_df["FinBERT_Label"].tolist()

predictions = []

for neighbors in I:

    labels = [
        train_labels[idx]
        for idx in neighbors
    ]

    prediction = max(
        set(labels),
        key=labels.count
    )

    predictions.append(prediction)

y_true = test_df["FinBERT_Label"].tolist()

acc = accuracy_score(
    y_true,
    predictions
)

print("FAISS BASELINE")

print(f"Accuracy: {acc:.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        predictions
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_true,
        predictions
    )
)

Train: (13514, 4)
Test : (3379, 4)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/212 [00:00<?, ?it/s]

Vectors Stored: 13514


Batches:   0%|          | 0/53 [00:00<?, ?it/s]


FAISS BASELINE
Accuracy: 0.7049

Classification Report:

              precision    recall  f1-score   support

    negative       0.68      0.68      0.68      1014
     neutral       0.78      0.63      0.70      1124
    positive       0.68      0.79      0.73      1241

    accuracy                           0.70      3379
   macro avg       0.71      0.70      0.70      3379
weighted avg       0.71      0.70      0.70      3379


Confusion Matrix:

[[694  97 223]
 [174 708 242]
 [155 106 980]]
